#### 示例 2：目标检测（保留列表）

每个样本是 `(image, boxes)`，其中 `boxes` 是一个 `(N, 4)` 的张量，每张图的 N 不同。

```python
def detection_collate_fn(batch):
    images = []
    boxes_list = []
    for image, boxes in batch:
        images.append(image)
        boxes_list.append(boxes)
    # 图像可以堆叠，因为尺寸相同
    images = torch.stack(images, dim=0)
    # boxes 不能堆叠，保持为列表
    return images, boxes_list
```

#### 示例 3：处理字典类型样本

如果数据集返回的是字典 `{"input_ids": tensor, "label": int}`，collate 可以逐键处理。

```python
def dict_collate_fn(batch):
    # batch 是字典列表
    collated = {}
    for key in batch[0].keys():
        if isinstance(batch[0][key], torch.Tensor):
            # 张量堆叠
            collated[key] = torch.stack([sample[key] for sample in batch])
        else:
            # 其他类型转为列表
            collated[key] = [sample[key] for sample in batch]
    return collated
```

#### 示例 4：返回自定义 Batch 对象

为了让代码更清晰，可以定义一个 `Batch` 类，在 `collate_fn` 中返回它的实例。

```python
class Batch:
    def __init__(self, inputs, masks, labels):
        self.inputs = inputs
        self.masks = masks
        self.labels = labels
    
    def to(self, device):
        self.inputs = self.inputs.to(device)
        self.masks = self.masks.to(device)
        self.labels = self.labels.to(device)
        return self

def batch_collate_fn(batch):
    # ... 处理得到 padded, masks, labels
    return Batch(padded, masks, labels)
```

训练时：
```python
for batch in loader:
    batch = batch.to(device)
    output = model(batch.inputs, attention_mask=batch.masks)
    loss = criterion(output, batch.labels)
```

### 6.5 总结

| 要点 | 说明 |
|------|------|
| **角色** | `collate_fn` 是 DataLoader 的**打包规则** |
| **输入** | 一个 `list`，每个元素是单个样本（`Dataset.__getitem__` 返回的内容） |
| **输出** | 任何你想要的结构（张量元组、字典、自定义对象） |
| **默认行为** | 将张量堆叠（`torch.stack`），要求形状完全一致 |
| **自定义场景** | 变长序列、目标检测、多模态数据、返回自定义对象等 |
| **与迭代协议的关系** | 完美衔接数据迭代流：`Dataset` → 单个样本 → 列表 → `collate_fn` → 批次 |

# Python 迭代协议详解：Iterable、Iterator、Generator

本文把这三个概念彻底拆开、揉碎，并结合 Python 的迭代协议从底层讲清楚。这不仅是为了概念本身，也是为了能真正用好它们（比如自定义 Dataset 和 DataLoader）。

## 1. 可迭代对象（Iterable）

### 定义

只要一个对象实现了 `__iter__` 方法，并且该方法**返回一个迭代器对象**，那这个对象就是**可迭代对象（Iterable）**。

更形式化地说，Python 的迭代协议要求：
- **可迭代对象**：`__iter__(self)` -> 返回迭代器。
- **迭代器**：必须同时实现 `__iter__`（返回自身）和 `__next__`。

In [ ]:
from collections.abc import Iterable

print(isinstance([1,2,3], Iterable))   # True
print(isinstance("hello", Iterable))   # True
print(isinstance(123, Iterable))       # False

内置函数 `iter(obj)` 本质上就是调用 `obj.__iter__()`，如果对象没有该方法就抛出 `TypeError`。所以 `iter(obj)` 能成功，就意味着它是可迭代对象。

### 常见的可迭代对象

- **序列**：列表、元组、字符串、range 对象
- **集合**：字典、集合、frozenset
- **文件对象**（逐行迭代）
- **许多第三方库的对象**（如 `torch.utils.data.Dataset`）

### 可迭代对象本身并不负责迭代状态

比如一个列表 `[1,2,3]` 是可迭代的，但列表本身**不记录**"现在取到第几个元素了"。列表仅仅具备返回一个迭代器的能力。真正的迭代状态保存在迭代器里。

所以，可迭代对象可以反复遍历，因为每一次调用 `iter()` 都会生成一个**全新的迭代器**，从头开始。

In [ ]:
lst = [1, 2, 3]
for x in lst: 
    print(x, end=' ')   # 第一次遍历：1 2 3
print()
for x in lst: 
    print(x, end=' ')   # 第二次遍历，没问题，因为重新获取了迭代器

## 2. 迭代器（Iterator）

### 定义

迭代器是**同时实现**了 `__iter__` 和 `__next__` 方法的对象。
- `__iter__`：返回迭代器对象自身（通常是 `return self`）。
- `__next__`：返回下一个可用的元素，如果元素耗尽则抛出 `StopIteration` 异常。

因为迭代器也有 `__iter__`，它本身也是可迭代对象。这就是为什么我们可以把迭代器直接放到 `for` 循环里。

### 内部状态

迭代器是有状态的，它必须记住当前迭代到哪个位置了。每次 `next(it)` 调用都会使内部游标向前移动，且不可后退。当耗尽时，继续调用只会抛出 `StopIteration`。

### 手动实现一个迭代器

In [ ]:
class CountDown:
    """倒数计时器迭代器"""
    
    def __init__(self, start):
        self.current = start

    def __iter__(self):
        return self   # 自身就是迭代器

    def __next__(self):
        if self.current <= 0:
            raise StopIteration
        num = self.current
        self.current -= 1
        return num

cd = CountDown(3)
for n in cd:
    print(n)   # 输出：3, 2, 1

这里 `CountDown` 的实例既是可迭代对象又是迭代器（因为 `__iter__` 返回自身）。但注意，**这样的迭代器只能遍历一次**，遍历完后内部状态已经耗尽，无法重置。

### 列表的迭代器是独立的

In [ ]:
lst = [1, 2, 3]
it1 = iter(lst)
it2 = iter(lst)

print(next(it1))  # 1
print(next(it1))  # 2
print(next(it2))  # 1 - it2 是独立的迭代器，从头开始

这体现了"可迭代对象"和"迭代器"解耦的好处：同一个数据源可以产生多个独立的迭代器。

### 关于"先有迭代器还是先有可迭代对象"

这个问题需要从设计角度看：

从**概念**上讲，先有"可迭代"这个抽象要求："一个东西能够被 for 循环遍历"，然后我们为了实现这种能力，设计了**迭代器**这个辅助对象来管理遍历状态。因此，逻辑上可迭代对象是接口，迭代器是具体实现手段。

Python 协议层面，可迭代对象通过 `__iter__` 交出迭代器，依赖迭代器来完成遍历。不存在"先有迭代器再有可迭代对象"的先后顺序，它们是相互配合的协议。

应该说："可迭代对象要求能返回一个迭代器"，迭代器是为可迭代对象服务的。

## 3. 生成器（Generator）

### 定义

生成器是一种**特殊的迭代器**，可以通过以下两种方式创建：
- **生成器函数**：使用 `yield` 关键字的函数。调用该函数会返回一个生成器对象，而不执行函数体。
- **生成器表达式**：类似列表推导式，但使用圆括号，例如 `(x*x for x in range(10))`。

生成器对象自动实现了 `__iter__` 和 `__next__`，因此它既是迭代器，也是可迭代对象。

### yield 如何工作

当生成器函数被调用时，它返回一个生成器对象，函数体内的代码此时不执行。第一次调用 `next(gen)` 时，代码开始执行，直到遇到 `yield value`，函数暂停并返回 `value`，同时保存所有局部状态（变量、指令指针等）。下一次调用 `next()` 时，从暂停处继续执行，直到再次遇到 `yield` 或函数结束（隐式抛出 `StopIteration`）。

In [ ]:
def my_gen():
    print("开始")
    yield 1
    print("中间")
    yield 2
    print("结束")

g = my_gen()
print(next(g))  # 打印"开始"，返回 1
print(next(g))  # 打印"中间"，返回 2
print(next(g))  # 打印"结束"，然后抛出 StopIteration

生成器函数可以包含多个 `yield`，甚至可以无限循环（例如一个不断产生数据的流水线）。

### 生成器表达式

In [ ]:
squares = (x*x for x in range(10))
print(next(squares))  # 0
print(next(squares))  # 1

它同样是一个惰性求值的迭代器，内存友好。

### 生成器的核心优势

- **简洁**：不需要显式定义类，不用手动维护 `self.current` 等状态。
- **自动实现协议**：yield 使得函数变成一个生成器对象，该对象天然就是迭代器。
- **天然惰性**：只按需产生值，适合处理大数据流或无限序列。

## 4. 三者关系总结

### 包含关系图

```
可迭代对象 (Iterable)          ← 只要实现了 __iter__ 的对象
    │
    └── 迭代器 (Iterator)      ← 同时实现了 __iter__ 和 __next__ 的对象
            │
            └── 生成器 (Generator)  ← 通过 yield 定义的函数或生成器表达式创建
```

用集合语言说：
- **所有生成器都是迭代器**。
- **所有迭代器都是可迭代对象**。
- 但并非所有可迭代对象都是迭代器（例如列表）。
- 并非所有迭代器都是生成器（例如通过类定义的迭代器）。

### 对比表

| 特性 | 可迭代对象 | 迭代器 | 生成器 |
|------|-----------|--------|--------|
| 必须方法 | `__iter__` | `__iter__`, `__next__` | 自动拥有（由 yield 实现） |
| 是否记录遍历状态 | 否 | 是 | 是 |
| 能否多次遍历 | 是（每次获取新迭代器） | 否（一次性） | 否（一次性） |
| 内存效率 | 取决于具体实现 | 惰性 | 惰性 |
| 创建难度 | 低（只需返回一个迭代器） | 中等（需手动维护状态） | 低（yield 即可） |

### 设计思想

可迭代对象是"遍历能力"的契约，迭代器是"如何遍历并保持位置"的状态机，生成器则是实现状态机的最便捷语法。

在实际编码中，当你需要自定义一个可迭代的数据源时，最简单的做法就是编写一个生成器函数，将其 `__iter__` 方法定义为生成器，从而让你的对象成为可迭代对象。

### 实战示例：分批数据加载器

In [ ]:
class BatchLoader:
    """分批数据加载器 - 使用生成器实现 __iter__"""
    
    def __init__(self, data, batch_size):
        self.data = data
        self.batch_size = batch_size

    def __iter__(self):                     
        """使用生成器返回批次，使对象成为可迭代对象"""
        for i in range(0, len(self.data), self.batch_size):
            yield self.data[i:i+self.batch_size]

# 使用示例
data = list(range(10))
loader = BatchLoader(data, batch_size=3)

for batch in loader:
    print(batch)  # [0,1,2] [3,4,5] [6,7,8] [9]

In [ ]:
import torch
from torch.nn.utils.rnn import pad_sequence

def my_collate_fn(batch):
    """
    处理变长序列的 collate_fn 示例
    
    batch: [(sentence_ids1, label1), (sentence_ids2, label2), ...]
    返回：(padded_sentences, attention_masks, labels_tensor)
    """
    # 解包成两个元组
    sentences, labels = zip(*batch)
    
    # 将句子列表转换为张量列表
    sentence_tensors = [torch.tensor(s) for s in sentences]
    
    # 填充到相同长度，batch_first=True 返回 (batch_size, max_len)
    padded_sentences = pad_sequence(sentence_tensors, batch_first=True, padding_value=0)
    
    # 生成 attention mask（1 表示真实 token，0 表示 padding）
    attention_masks = (padded_sentences != 0).long()
    
    # 标签直接堆叠
    labels_tensor = torch.tensor(labels)
    
    return padded_sentences, attention_masks, labels_tensor

# 使用
loader = DataLoader(dataset, batch_size=16, collate_fn=my_collate_fn)

这样 `BatchLoader` 是可迭代对象，它的 `__iter__` 返回一个生成器（迭代器）。每次 `for batch in loader` 都会创建一个新的生成器，因此可以多次遍历。

---

## 5. 与 PyTorch DataLoader 的联系

在 PyTorch 中，`DataLoader` 的底层实现就依赖这套迭代协议：

```python
from torch.utils.data import Dataset, DataLoader

class MyDataset(Dataset):
    # Dataset 是可迭代对象，需要实现 __len__ 和 __getitem__
    pass

dataset = MyDataset()
loader = DataLoader(dataset, batch_size=32)

# DataLoader 返回的其实就是迭代器
for batch in loader:  # 内部调用 iter(loader) -> 迭代器 -> next() 取数据
    pass
```

理解这套协议，能帮助你自定义更灵活的数据加载逻辑。

---

## 6. DataLoader 中的 collate_fn：批次打包规则

### 6.1 collate_fn 在数据流中的位置

回忆一下 DataLoader 的工作流程：

```
Dataset → 单个样本 → [样本 1, 样本 2, ..., 样本 N] → collate_fn → 批次数据
```

`collate_fn` 就是**第二步到第三步之间的"打包员"**，它定义了如何将多个单个样本打包成一个批次（batch）。

**工作流程**：
1. 从 `Dataset`（或其迭代器）中逐个取出**单个样本**
2. 攒够 `batch_size` 个样本，把它们组成一个**列表**
3. 把这个列表传给 `collate_fn` 函数，该函数返回一个**合并后的批次数据**
4. 训练循环中拿到的就是这个批次的返回值